In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


REPO_ID = "witgaw/METR-LA"
REPO_TYPE = "dataset"

ZERO_THRESHOLD = 1.0
DENSITY_THRESHOLD = 1000.0

print("Notebook 06: Sensor-Level Multivariate Analysis")
print("Repository :", REPO_ID)

In [ ]:
train = load_dataset(
    REPO_ID,
    split="train"
)

print(train)

print("\nRows    :", train.num_rows)
print("Columns :", len(train.column_names))

assert train.num_rows > 0
assert "node_id" in train.column_names
assert "t0_timestamp" in train.column_names

sensor_count = train.unique("node_id")

print("Sensors :", len(sensor_count))

assert len(sensor_count) == 207

print("\nPASS: METR-LA loaded.")

In [ ]:
if "x_t+0_d0" in train.column_names:
    CURRENT_SPEED = "x_t+0_d0"

elif "x_t-0_d0" in train.column_names:
    CURRENT_SPEED = "x_t-0_d0"

else:
    raise AssertionError(
        "Could not locate current-time d0 speed column."
    )


NEXT_SPEED = "y_t+1_d0"

assert NEXT_SPEED in train.column_names


print("Current speed :", CURRENT_SPEED)
print("Next speed    :", NEXT_SPEED)

print("\nPASS: Forecast columns identified.")

In [ ]:
base_df = (
    train
    .select_columns([
        "node_id",
        "t0_timestamp",
        CURRENT_SPEED,
        NEXT_SPEED
    ])
    .to_pandas()
)

base_df["node_id"] = (
    base_df["node_id"].astype(int)
)

base_df["t0_timestamp"] = pd.to_datetime(
    base_df["t0_timestamp"]
)

base_df = base_df.rename(
    columns={
        CURRENT_SPEED: "speed",
        NEXT_SPEED: "next_speed"
    }
)

print("Rows       :", len(base_df))
print("Sensors    :", base_df["node_id"].nunique())
print("Timestamps :", base_df["t0_timestamp"].nunique())

assert base_df["node_id"].nunique() == 207
assert base_df["speed"].notna().all()
assert base_df["next_speed"].notna().all()

print("\nPASS: Base sensor-time table created.")

In [ ]:
base_df["persistence_error"] = np.abs(
    base_df["next_speed"]
    - base_df["speed"]
)

disparity_df = (
    base_df
    .groupby("node_id", as_index=False)
    .agg(
        disparity=(
            "persistence_error",
            "mean"
        )
    )
)

print(disparity_df.head().to_string(index=False))

print(
    "\nMean persistence MAE:",
    disparity_df["disparity"].mean()
)

assert len(disparity_df) == 207
assert disparity_df["node_id"].is_unique
assert disparity_df["disparity"].notna().all()
assert (disparity_df["disparity"] >= 0).all()

print("\nPASS: Per-sensor forecast disparity reconstructed.")

In [ ]:
base_df["near_zero"] = (
    base_df["speed"] <= ZERO_THRESHOLD
)

failure_df = (
    base_df
    .groupby("node_id", as_index=False)
    .agg(
        zero_rate=(
            "near_zero",
            "mean"
        )
    )
)

overall_zero_rate = (
    base_df["near_zero"].mean()
)

print(
    f"Overall near-zero rate: "
    f"{overall_zero_rate:.4%}"
)

print("\nPer-sensor example:")

print(
    failure_df.head().to_string(index=False)
)

assert len(failure_df) == 207
assert failure_df["zero_rate"].between(0, 1).all()

print("\nPASS: Failure rate reconstructed.")

In [ ]:
def detect_cusum(values, threshold=5.0, drift=0.5):

    values = np.asarray(values, dtype=float)

    mean = np.mean(values)
    std = np.std(values)

    if std == 0:
        return 0

    z = (values - mean) / std

    positive = 0.0
    negative = 0.0
    flags = 0

    for value in z:

        positive = max(
            0,
            positive + value - drift
        )

        negative = min(
            0,
            negative + value + drift
        )

        if (
            positive > threshold
            or negative < -threshold
        ):

            flags += 1

            positive = 0.0
            negative = 0.0

    return flags


def detect_ewma(
    values,
    alpha=0.2,
    control_limit=3.0
):

    values = np.asarray(values, dtype=float)

    mean = np.mean(values)
    std = np.std(values)

    if std == 0:
        return 0

    ewma = mean
    flags = 0

    ewma_std = (
        std
        * np.sqrt(
            alpha / (2 - alpha)
        )
    )

    upper = (
        mean
        + control_limit * ewma_std
    )

    lower = (
        mean
        - control_limit * ewma_std
    )

    for value in values:

        ewma = (
            alpha * value
            + (1 - alpha) * ewma
        )

        if (
            ewma > upper
            or ewma < lower
        ):
            flags += 1

    return flags


print("CUSUM and EWMA functions defined.")

In [ ]:
drift_results = []

grouped = base_df.groupby(
    "node_id",
    sort=True
)

for node_id, group in grouped:

    group = group.sort_values(
        "t0_timestamp"
    )

    values = (
        group["speed"]
        .to_numpy(dtype=float)
    )

    cusum_flags = detect_cusum(
        values
    )

    ewma_flags = detect_ewma(
        values
    )

    drift_results.append({
        "node_id": int(node_id),
        "cusum_flags": cusum_flags,
        "ewma_flags": ewma_flags,
        "observations": len(values)
    })


drift_df = pd.DataFrame(
    drift_results
)

print("Sensors analyzed:", len(drift_df))

print(
    drift_df.head().to_string(index=False)
)

print(
    "\nTotal CUSUM flags:",
    drift_df["cusum_flags"].sum()
)

print(
    "Total EWMA flags:",
    drift_df["ewma_flags"].sum()
)

assert len(drift_df) == 207
assert drift_df["node_id"].is_unique

print("\nPASS: Drift detection completed.")

In [ ]:
reliability_df = (
    failure_df
    .merge(
        drift_df,
        on="node_id",
        how="inner"
    )
)

reliability_df["cusum_rate"] = (
    reliability_df["cusum_flags"]
    / reliability_df["observations"]
)

reliability_df["ewma_rate"] = (
    reliability_df["ewma_flags"]
    / reliability_df["observations"]
)


# Simple weighted reliability score
reliability_df["reliability"] = (
    1.0
    - (
        0.50 * reliability_df["zero_rate"]
        + 0.25 * reliability_df["cusum_rate"]
        + 0.25 * reliability_df["ewma_rate"]
    )
)

reliability_df["reliability"] = (
    reliability_df["reliability"]
    .clip(0, 1)
)


print(
    reliability_df[
        [
            "node_id",
            "zero_rate",
            "cusum_rate",
            "ewma_rate",
            "reliability"
        ]
    ]
    .head()
    .to_string(index=False)
)

assert len(reliability_df) == 207
assert reliability_df["reliability"].between(0, 1).all()

print("\nPASS: Reliability score reconstructed.")

In [ ]:
repo_files = list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE
)

required_graph_files = [
    "adj_mx.npy",
    "adj_mx_mapping.json",
    "distances.csv",
    "sensor_locations.csv"
]

file_paths = {}

for filename in required_graph_files:

    matches = [
        path
        for path in repo_files
        if os.path.basename(path) == filename
    ]

    print(filename, "->", matches)

    assert len(matches) == 1, (
        f"Could not uniquely find {filename}"
    )

    file_paths[filename] = matches[0]


print("\nPASS: Graph files discovered.")

In [ ]:
local_files = {}

for filename, repo_path in file_paths.items():

    local_files[filename] = hf_hub_download(
        repo_id=REPO_ID,
        filename=repo_path,
        repo_type=REPO_TYPE
    )


adj_mx = np.load(
    local_files["adj_mx.npy"]
)

with open(
    local_files["adj_mx_mapping.json"],
    "r",
    encoding="utf-8"
) as f:

    mapping = json.load(f)


distances = pd.read_csv(
    local_files["distances.csv"]
)

locations = pd.read_csv(
    local_files["sensor_locations.csv"]
)


print("Adjacency :", adj_mx.shape)
print("Distances :", distances.shape)
print("Locations :", locations.shape)
print("Mapping   :", type(mapping).__name__)

assert adj_mx.shape == (207, 207)

print("\nPASS: Graph files loaded.")

In [ ]:
adj_no_self = adj_mx.copy()

np.fill_diagonal(
    adj_no_self,
    0
)

degrees = np.count_nonzero(
    adj_no_self,
    axis=1
)

topology_df = pd.DataFrame({
    "node_id": np.arange(207),
    "topology_sensitivity_proxy": degrees
})


print("=" * 65)
print("TOPOLOGY FEATURE NOTE")
print("=" * 65)

print(
    "Actual topology sensitivity from graph "
    "edge-dropout has NOT been computed yet."
)

print(
    "Node degree is used only as a PLACEHOLDER proxy."
)

print(
    "Replace this feature with forecast-error degradation "
    "after the edge-dropout experiment."
)

print("\nDegree summary:")
print(
    topology_df[
        "topology_sensitivity_proxy"
    ].describe()
)

assert len(topology_df) == 207
assert topology_df["node_id"].is_unique

print("\nPASS: Topology proxy created.")

In [ ]:
print("Distance columns:")
print(distances.columns.tolist())

print("\nFirst 5 rows:")
print(
    distances.head().to_string(index=False)
)

assert {
    "from",
    "to",
    "cost"
}.issubset(
    distances.columns
)

SOURCE_COL = "from"
TARGET_COL = "to"
DIST_COL = "cost"

print("\nPASS: Distance schema confirmed.")

In [ ]:
distance_work = distances[
    [
        SOURCE_COL,
        TARGET_COL,
        DIST_COL
    ]
].copy()

distance_work = distance_work[
    (
        distance_work[SOURCE_COL]
        != distance_work[TARGET_COL]
    )
    &
    (
        distance_work[DIST_COL] > 0
    )
]


nearby_pairs = distance_work[
    distance_work[DIST_COL]
    <= DENSITY_THRESHOLD
].copy()


all_distance_ids = set(
    distances[SOURCE_COL]
).union(
    set(distances[TARGET_COL])
)


neighbors = {
    sensor_id: set()
    for sensor_id in all_distance_ids
}


for source, target in zip(
    nearby_pairs[SOURCE_COL],
    nearby_pairs[TARGET_COL]
):

    neighbors[source].add(target)
    neighbors[target].add(source)


density_physical = pd.DataFrame({
    "physical_sensor_id":
        list(neighbors.keys()),

    "density":
        [
            len(neighbors[sensor_id])
            for sensor_id in neighbors
        ]
})


print(
    "Physical IDs in distance file:",
    len(density_physical)
)

print(
    density_physical.head().to_string(
        index=False
    )
)

assert len(density_physical) > 0

print("\nPASS: Physical-ID density calculated.")

In [ ]:
print("=" * 65)
print("DENSITY ↔ GRAPH IDENTIFIER CHECK")
print("=" * 65)

print("Mapping type:", type(mapping).__name__)

if isinstance(mapping, dict):

    print("Mapping keys:")
    print(list(mapping.keys()))

    if "sensor_ids" in mapping:

        graph_sensor_ids = (
            pd.Series(
                mapping["sensor_ids"]
            )
            .astype(str)
        )

        print(
            "\nGraph sensor IDs:",
            len(graph_sensor_ids)
        )

        print(
            "First 10:",
            graph_sensor_ids.head(10).tolist()
        )

else:

    graph_sensor_ids = None


distance_ids_string = set(
    density_physical[
        "physical_sensor_id"
    ].astype(str)
)


if graph_sensor_ids is not None:

    overlap = set(
        graph_sensor_ids
    ).intersection(
        distance_ids_string
    )

    print(
        "\nDirect graph ↔ distance ID matches:",
        len(overlap)
    )

In [ ]:
density_aligned = None
density_alignment_method = None


# --------------------------------------------------
# Case 1: distance IDs directly equal graph physical IDs
# --------------------------------------------------

if (
    graph_sensor_ids is not None
    and len(overlap) == 207
):

    graph_id_map = pd.DataFrame({
        "node_id": np.arange(207),
        "_join_id": graph_sensor_ids
    })

    density_temp = density_physical.copy()

    density_temp["_join_id"] = (
        density_temp[
            "physical_sensor_id"
        ].astype(str)
    )

    density_aligned = (
        graph_id_map
        .merge(
            density_temp[
                ["_join_id", "density"]
            ],
            on="_join_id",
            how="left"
        )
        [
            ["node_id", "density"]
        ]
    )

    density_alignment_method = (
        "physical sensor ID mapping"
    )


# --------------------------------------------------
# Case 2: density file itself uses 0...206
# --------------------------------------------------

else:

    numeric_density_ids = set(
        pd.to_numeric(
            density_physical[
                "physical_sensor_id"
            ],
            errors="coerce"
        )
        .dropna()
        .astype(int)
    )

    expected_node_ids = set(
        range(207)
    )

    if expected_node_ids.issubset(
        numeric_density_ids
    ):

        density_aligned = (
            density_physical
            .rename(
                columns={
                    "physical_sensor_id":
                        "node_id"
                }
            )
            [
                ["node_id", "density"]
            ]
        )

        density_aligned["node_id"] = (
            density_aligned["node_id"]
            .astype(int)
        )

        density_alignment_method = (
            "direct node_id"
        )


if density_aligned is None:

    print("\nWARNING:")
    print(
        "distances.csv and adj_mx_mapping.json "
        "do not expose the same sensor identifier system."
    )

    print(
        "Density cannot safely be attached to "
        "node_id 0...206 from these files alone."
    )

    print(
        "\nDo NOT merge density by row position."
    )

else:

    print(
        "\nDensity alignment:",
        density_alignment_method
    )

    print(
        "Matched densities:",
        density_aligned["density"].notna().sum()
    )

In [ ]:
assert density_aligned is not None, (
    "Density IDs are not safely aligned with graph node_id."
)

reliability_feature = reliability_df[
    ["node_id", "reliability"]
].copy()


sensor_features = (
    reliability_feature
    .merge(
        density_aligned,
        on="node_id",
        how="outer"
    )
    .merge(
        topology_df,
        on="node_id",
        how="outer"
    )
    .merge(
        disparity_df,
        on="node_id",
        how="outer"
    )
)


feature_cols = [
    "reliability",
    "density",
    "topology_sensitivity_proxy",
    "disparity"
]


complete_mask = (
    sensor_features[
        feature_cols
    ]
    .notna()
    .all(axis=1)
)


matched_count = int(
    complete_mask.sum()
)


print("=" * 65)
print("FOUR-SOURCE ALIGNMENT")
print("=" * 65)

print("Total sensors     :", len(sensor_features))
print("Four-way matches  :", matched_count)

print("\nMissing values:")

print(
    sensor_features[
        feature_cols
    ].isna().sum()
)


unmatched = sensor_features[
    ~complete_mask
]


if unmatched.empty:

    print(
        "\nPASS: All sensors matched."
    )

else:

    print(
        f"\nWARNING: {len(unmatched)} "
        "sensor(s) failed to join."
    )

    print(
        unmatched.to_string(index=False)
    )

In [ ]:
analysis_df = (
    sensor_features[
        complete_mask
    ]
    [
        ["node_id"] + feature_cols
    ]
    .sort_values("node_id")
    .reset_index(drop=True)
)


print("Analysis sensors:", len(analysis_df))

print(
    analysis_df.head().to_string(
        index=False
    )
)

assert len(analysis_df) >= 0.95 * 207

assert analysis_df["node_id"].is_unique

assert (
    analysis_df[feature_cols]
    .notna()
    .all()
    .all()
)

print("\nPASS: Analysis table ready.")

In [ ]:
corr_cols = [
    "reliability",
    "density",
    "topology_sensitivity_proxy",
    "disparity"
]

corr_matrix = (
    analysis_df[corr_cols]
    .corr(method="pearson")
)


print("=" * 65)
print("PAIRWISE CORRELATION MATRIX")
print("=" * 65)

print(
    corr_matrix.round(4).to_string()
)

assert corr_matrix.shape == (4, 4)

In [ ]:
labels = [
    "Reliability",
    "Density",
    "Topology Proxy",
    "Disparity"
]

fig, ax = plt.subplots(
    figsize=(9, 7)
)

image = ax.imshow(
    corr_matrix,
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    range(4),
    labels=labels,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    range(4),
    labels=labels
)


for i in range(4):

    for j in range(4):

        ax.text(
            j,
            i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )


cbar = plt.colorbar(
    image,
    ax=ax
)

cbar.set_label(
    "Pearson correlation"
)

ax.set_title(
    "Sensor-Level Feature Correlations"
)

plt.tight_layout()
plt.show()

In [ ]:
predictors = [
    "reliability",
    "density",
    "topology_sensitivity_proxy"
]

predictor_corr = (
    analysis_df[predictors]
    .corr()
)

HIGH_CORR_THRESHOLD = 0.70

high_corr_pairs = []


for i in range(len(predictors)):

    for j in range(i + 1, len(predictors)):

        p1 = predictors[i]
        p2 = predictors[j]

        r = predictor_corr.loc[
            p1,
            p2
        ]

        if abs(r) > HIGH_CORR_THRESHOLD:

            high_corr_pairs.append(
                (p1, p2, r)
            )


print("=" * 65)
print("PREDICTOR MULTICOLLINEARITY CHECK")
print("=" * 65)

print(
    predictor_corr.round(4).to_string()
)


if len(high_corr_pairs) == 0:

    print(
        "\nPASS: No predictor pair has |r| > 0.70."
    )

else:

    print(
        "\nWARNING: High predictor correlation detected."
    )

    for p1, p2, r in high_corr_pairs:

        print(
            f"{p1} <-> {p2}: r = {r:.4f}"
        )

    print(
        "\nIndividual contributions may be difficult "
        "to separate statistically."
    )

In [ ]:
X = (
    analysis_df[predictors]
    .astype(float)
)

y = (
    analysis_df["disparity"]
    .astype(float)
)

X_const = sm.add_constant(X)


ols_model = sm.OLS(
    y,
    X_const
).fit()


print(
    ols_model.summary()
)

In [ ]:
vif_results = pd.DataFrame({
    "predictor": predictors,

    "VIF": [
        variance_inflation_factor(
            X.values,
            i
        )
        for i in range(
            X.shape[1]
        )
    ]
})


print("=" * 65)
print("VARIANCE INFLATION FACTORS")
print("=" * 65)

print(
    vif_results.round(4).to_string(
        index=False
    )
)


high_vif = vif_results[
    vif_results["VIF"] > 5
]


if high_vif.empty:

    print(
        "\nPASS: No predictor has VIF > 5."
    )

else:

    print(
        "\nWARNING: VIF > 5 detected."
    )

    print(
        high_vif.to_string(
            index=False
        )
    )

    print(
        "\nMulticollinearity may make individual "
        "coefficient estimates unstable."
    )

In [ ]:
correlation_ok = (
    len(high_corr_pairs) == 0
)

vif_ok = (
    high_vif.empty
)

variation_ok = (
    analysis_df[predictors]
    .nunique()
    .gt(1)
    .all()
)

alignment_ok = (
    len(analysis_df)
    >= 0.95 * 207
)


print("=" * 70)
print("ATTRIBUTION READINESS SUMMARY")
print("=" * 70)

print(
    "Sensor alignment       :",
    "PASS" if alignment_ok else "WARNING"
)

print(
    "Predictor variation    :",
    "PASS" if variation_ok else "WARNING"
)

print(
    "|Correlation| <= 0.70  :",
    "PASS" if correlation_ok else "WARNING"
)

print(
    "VIF <= 5              :",
    "PASS" if vif_ok else "WARNING"
)

print()


if (
    alignment_ok
    and variation_ok
    and correlation_ok
    and vif_ok
):

    print(
        "CONCLUSION:"
    )

    print(
        "The current predictors contain enough independent "
        "statistical variation to proceed to a more rigorous "
        "attribution analysis."
    )

else:

    print(
        "CONCLUSION:"
    )

    print(
        "The current feature set is not yet sufficiently "
        "independent for reliable individual attribution."
    )

    print(
        "Feature redesign, transformation, or replacement "
        "should be considered first."
    )


print(
    "\nIMPORTANT: This result does NOT establish causality."
)

print(
    "Low correlation and acceptable VIF only indicate that "
    "multicollinearity is not preventing further analysis."
)

print(
    "\nTOPOLOGY WARNING: Node degree is currently only a "
    "placeholder. Replace it with actual edge-dropout "
    "forecast-error degradation before final attribution."
)

In [ ]:
tests = {
    "207 reliability scores":
        len(reliability_df) == 207,

    "207 disparity scores":
        len(disparity_df) == 207,

    "207 topology proxies":
        len(topology_df) == 207,

    "Density safely aligned":
        density_aligned is not None,

    "At least 95% four-way sensor match":
        len(analysis_df) >= 0.95 * 207,

    "Correlation matrix complete":
        corr_matrix.shape == (4, 4),

    "Predictor correlation checked":
        predictor_corr.shape == (3, 3),

    "OLS regression completed":
        ols_model is not None,

    "Three VIF values calculated":
        len(vif_results) == 3
}


print("=" * 65)
print("MULTIVARIATE SENSOR ANALYSIS VALIDATION")
print("=" * 65)

for name, passed in tests.items():

    print(
        f"{'PASS' if passed else 'FAIL':4} : {name}"
    )

print("=" * 65)


print(f"""
Sensors analyzed           : {len(analysis_df)}
Four-way matched sensors   : {matched_count}

Reliability                : Zero + CUSUM + EWMA
Density                    : Neighbors within 1000 m
Topology sensitivity       : NODE DEGREE [PLACEHOLDER]
Forecast disparity         : Persistence MAE

High-correlation pairs     : {len(high_corr_pairs)}
Predictors with VIF > 5    : {len(high_vif)}

OLS R-squared              : {ols_model.rsquared:.4f}
OLS adjusted R-squared     : {ols_model.rsquared_adj:.4f}
""")


if all(tests.values()):

    print("ALL TEST CASES PASSED")

else:

    failed = [
        name
        for name, passed in tests.items()
        if not passed
    ]

    print("FAILED TESTS:")

    for name in failed:
        print(" -", name)

    raise AssertionError(
        f"{len(failed)} validation test(s) failed."
    )